# Gemma 4 Finance QLoRA Fine-Tune
Run on Kaggle T4. Before running:
1. Add your HuggingFace token as a Kaggle secret named `HF_TOKEN`
2. Upload `combined_train.jsonl` and `combined_val.jsonl` as a Kaggle dataset input (e.g. named `finance-ft`)
3. Accept Gemma 4 license at huggingface.co/google/gemma-4-E2B-it

In [ ]:
# Install — do NOT pin transformers separately
!pip install unsloth trl datasets -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-4-E2B-it',
    max_seq_length=2048,
    load_in_4bit=True,
    full_finetuning=False,
    dtype=None,
    token=hf_token,
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

# Adjust path to match your Kaggle dataset input name
train_ds = load_dataset('json', data_files='/kaggle/input/finance-ft/combined_train.jsonl', split='train')
val_ds   = load_dataset('json', data_files='/kaggle/input/finance-ft/combined_val.jsonl',   split='train')

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir='/kaggle/working/checkpoints',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,           # T4 has no bfloat16 support
        bf16=False,
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        max_seq_length=2048,
        packing=False,       # Packing corrupts <think> block boundaries
        dataset_text_field=None,
    ),
)

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# Save LoRA adapter only (~50-100MB) — avoids Kaggle output size limit
# To get the merged full model later, run locally: model.merge_and_unload()
model.save_pretrained('/kaggle/working/gemma-finance-lora')
tokenizer.save_pretrained('/kaggle/working/gemma-finance-lora')
print('Adapter saved to /kaggle/working/gemma-finance-lora')